Updated and stored your preferred learning format.

🚀 Great. You've completed:

Day 1 → Data understanding + merge + popularity baseline
Day 2 → Weighted ratings + popularity filtering
Day 3 → User-based collaborative filtering
Day 4 → Item-based collaborative filtering
Day 5 → SVD / Matrix Factorization
Day 6 → Real Top-N recommendation engine

Now we move toward solving one of the biggest problems in recommendation systems:

"How do we improve recommendation quality?"

Right now your system only uses ratings.

Real systems combine multiple signals:

User ratings

Movie genres

Similar content

User history

Popularity

Time behavior


So now we start adding another brain to the system.

🚀 Day 7 — Content-Based Recommendation System

Until now:

Collaborative filtering:

User A liked X
User B liked X

→ Maybe User B likes Y

Problem:

Collaborative filtering struggles when:

new users arrive

new movies arrive

little rating data exists


This is called:

Cold Start Problem

Content-based recommendation helps solve it.


---

🧠 Logic

Recommend based on movie characteristics

Example:

User likes:

Batman
Spiderman
Avengers

System notices:

Action
Superhero
Adventure

Then recommends:

Iron Man
Captain America

instead of random movies.


---

Step 1 — Create movie features

MovieLens has genre columns:

Action
Comedy
Drama
Sci-Fi
Romance
...

Combine them into one text feature.

genre_columns = movies.columns[5:]

movies['genres'] = movies[
    genre_columns
].apply(
    lambda x:' '.join(
        x.index[x==1]
    ),
    axis=1
)

movies[
    ['title','genres']
].head()

Expected output:

Toy Story      Animation Comedy Children's
Batman         Action Adventure
Titanic        Romance Drama


---

Step 2 — Convert text into numbers

Machines cannot understand:

Action Comedy Romance

Need numerical vectors.

Use:

from sklearn.feature_extraction.text import CountVectorizer

cv=CountVectorizer()

movie_vectors=cv.fit_transform(
    movies['genres']
)

What happens internally:

Action Comedy Romance
↓

[1,1,1,0,0...]

Action Drama
↓

[1,0,1,0,0...]


---

Step 3 — Compute similarity

from sklearn.metrics.pairwise import cosine_similarity

similarity=cosine_similarity(
    movie_vectors
)

Now every movie gets similarity scores with every other movie.

Example:

Batman

Toy Story → 0.12
Iron Man → 0.92
Titanic → 0.08


---

Step 4 — Recommendation function

def recommend(movie_name):

    movie_index=movies[
        movies['title']==movie_name
    ].index[0]

    distances=list(
        enumerate(
            similarity[movie_index]
        )
    )

    distances=sorted(
        distances,
        key=lambda x:x[1],
        reverse=True
    )

    for i in distances[1:6]:

        print(
            movies.iloc[
                i[0]
            ]['title']
        )


---

Step 5 — Test it

recommend(
    'Batman Forever (1995)'
)

Possible output:

Batman Returns
Batman
Superman
Judge Dredd
Robocop


---

🧠 What is happening?

Movie

Batman

↓

Find genre pattern

↓

Compare with all movies

↓

Recommend most similar movies


---

Important difference

Collaborative Filtering:

Uses users

Content-Based:

Uses item features


---

Advantages

✅ Works for new users

✅ Works for new movies

✅ Doesn't need many ratings


---

Disadvantages

❌ Recommends very similar items only

Example:

Batman
Batman Returns
Batman Forever
Batman Begins

You may miss unexpected discoveries.


---

🎯 Homework

1. Try:

recommend(
    'Toy Story (1995)'
)

recommend(
    'Titanic (1997)'
)

recommend(
    'Star Wars (1977)'
)

Check:

Are recommendations sensible?

Are they too similar?



---

2. Think about this:

Question:

If collaborative filtering says:

Titanic

and content-based says:

Avatar

How can we combine both?

That question leads directly to:

🚀 Day 8 — Hybrid Recommendation System



Yes — that's the right direction. Small refinement:

We're not really doing embeddings yet in Day 7. We are creating a vector representation (feature vector) of genres.

Current flow:

Toy Story
Genres:
Animation Comedy Children's

After CountVectorizer:

Action      Comedy     Romance     Animation    Children's
0              1            0             1             1

Batman:

Action      Comedy     Romance     Animation    Children's
1              0            0             0             0

Now similarity becomes:

Movie A vector ↔ Movie B vector

Cosine similarity compares them and asks:

> "How similar are these patterns?"



So your understanding becomes:

Genres
↓
Convert into numerical vectors
↓
Measure relationships/similarity
↓
Recommend similar movies

Embeddings are the next level of this idea.

What we are doing now:

Action = [1,0,0,0]
Comedy = [0,1,0,0]

Every genre is independent.

Real embeddings learn richer relationships automatically:

Action = [0.8,0.3,0.6]
Adventure = [0.75,0.35,0.55]
Romance = [0.1,0.9,0.2]

Now the model can learn:

Action ≈ Adventure

even if the words aren't identical.

Examples:

Word2Vec

BERT

movie embeddings from neural recommendation systems


So Day 7 is more like:

Feature encoding/vectorization

Later we'll reach:

Learned embeddings

That connection you made is important because SVD latent features from Day 5 were already behaving somewhat like learned embeddings, even though we didn't call them that at the time.


In [1]:
# Load dataset

from surprise import Dataset, Reader, SVD, accuracy

import pandas as pd

from surprise.model_selection import train_test_split

import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer

from sklearn.metrics.pairwise import cosine_similarity

In [2]:
ratings_cols = [ 'user_id', 'movie_id', 'rating', 'timestamp' ]

ratings = pd.read_csv ( '../data/ml-100k/u.data', sep = '\t', names = ratings_cols )

ratings.head()


,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [3]:
movie_cols = [
    'movie_id',
    'title',
    'release_date',
    'video_release',
    'IMDb_URL',
    'unknown',
    'Action',
    'Adventure',
    'Animation',
    'Children',
    'Comedy',
    'Crime',
    'Documentary',
    'Drama',
    'Fantasy',
    'Film_Noir',
    'Horror',
    'Musical',
    'Mystery',
    'Romance',
    'SciFi',
    'Thriller',
    'War',
    'Western'
]

movies = pd.read_csv(
    '../data/ml-100k/u.item',
    sep='|',
    encoding='latin-1',
    header=None,
    names=movie_cols
)

movies.head()


,movie_id,title,release_date,video_release,IMDb_URL,unknown,Action,Adventure,Animation,Children,...,Fantasy,Film_Noir,Horror,Musical,Mystery,Romance,SciFi,Thriller,War,Western
0,1,Toy Story (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [4]:
# Create a movie features and combine them

genre_columns = movies.columns [5:]

#print(movies[genre_columns].iloc[0])

movies['genres'] = movies [ genre_columns ].apply ( lambda x : ' '.join ( x.index [ x == 1 ] ), axis = 1 )

movies [ [ 'title', 'genres' ] ].head()

,title,genres
0,Toy Story (1995),Animation Children Comedy
1,GoldenEye (1995),Action Adventure Thriller
2,Four Rooms (1995),Thriller
3,Get Shorty (1995),Action Comedy Drama
4,Copycat (1995),Crime Drama Thriller


Alright, let’s break down that second line in detail:

```python
movies['genres'] = movies[genre_columns].apply(lambda x: ''.join(x.index[x == 1]), axis=1)
```

---

### Step‑by‑step explanation

1. **`movies[genre_columns]`**  
   - `genre_columns = movies.columns[5:]` means you selected all columns from the 6th onward.  
   - In the MovieLens dataset, those columns are typically genre indicator columns (e.g., `Action`, `Comedy`, `Drama`, etc.), where each cell is `0` or `1`.

   Example row:
   ```
   Action   Comedy   Drama   Romance
   1        0        1       0
   ```

2. **`.apply(..., axis=1)`**  
   - `.apply()` applies a function to each row (because `axis=1` means “row‑wise”).  
   - So for each movie, you pass its genre indicator row into the lambda function.

3. **`lambda x: ''.join(x.index[x == 1])`**  
   - `x` is a **Series** representing one row of genre indicators.  
   - `x == 1` produces a boolean mask: `True` where the genre is present, `False` otherwise.  
     Example:  
     ```
     Action   Comedy   Drama   Romance
     True     False    True    False
     ```
   - `x.index[x == 1]` selects the column names where the value is `1`.  
     Example result: `Index(['Action', 'Drama'], dtype='object')`
   - `''.join(...)` concatenates those genre names into a single string.  
     Example result: `"ActionDrama"`

4. **Assignment to `movies['genres']`**  
   - The new column `genres` is created in the `movies` DataFrame.  
   - Each row now contains a string of genres for that movie.

---

### ✅ Example in action
Suppose row 0 looks like:
```
Action=1, Comedy=0, Drama=1, Romance=0
```

- `x.index[x == 1]` → `['Action', 'Drama']`
- `''.join(...)` → `"ActionDrama"`
- Stored in `movies['genres'][0]`.

---

### 🎯 Concept
This line is **transforming one‑hot encoded genre columns into a single string label per movie**.  
Instead of having many binary columns, you now have a human‑readable genre string.

---

👉 One improvement: if you want genres separated by commas instead of mashed together, you can use:

```python
movies['genres'] = movies[genre_columns].apply(lambda x: ','.join(x.index[x == 1]), axis=1)
```

That way you’d get `"Action,Drama"` instead of `"ActionDrama`.  

Would you like me to show you how to **explode that string back into a list of genres per movie** (so you can easily filter movies by genre later)?

**One‑hot encoding is a preprocessing technique that converts categorical variables into binary columns, ensuring machine learning models can interpret them correctly without assuming any false order. Each category becomes its own column, marked with `1` if present and `0` if absent.**

---

## 🔎 Why One‑Hot Encoding is Needed
- **Categorical data** (like colors, cities, genres) cannot be directly used by most ML algorithms.
- If you assign numbers directly (e.g., Red=0, Blue=1, Green=2), the model may wrongly assume an order or magnitude relationship.
- **One‑hot encoding avoids this problem** by treating each category independently.

---

## ✅ How It Works
Suppose you have a dataset with a column `Color`:

| Item | Color  |
|------|--------|
| 1    | Red    |
| 2    | Blue   |
| 3    | Green  |
| 4    | Red    |

After one‑hot encoding, `Color` is replaced by three binary columns:

| Item | Color_Red | Color_Blue | Color_Green |
|------|-----------|------------|-------------|
| 1    | 1         | 0          | 0           |
| 2    | 0         | 1          | 0           |
| 3    | 0         | 0          | 1           |
| 4    | 1         | 0          | 0           |

- Each row has exactly one `1` in the corresponding category column.
- No false ordering is implied between Red, Blue, and Green.  [GeeksForGeeks](https://www.geeksforgeeks.org/machine-learning/ml-one-hot-encoding/)  [DataCamp](https://www.datacamp.com/tutorial/one-hot-encoding-python-tutorial)  [Udacity](https://www.udacity.com/blog/one-hot-encoding-explained-a-key-step-in-data-preprocessing/)

---

## ⚙️ Implementation in Python

### Using Pandas
```python
import pandas as pd

df = pd.DataFrame({'Item':[1,2,3,4], 'Color':['Red','Blue','Green','Red']})
encoded = pd.get_dummies(df, columns=['Color'])
print(encoded)
```

Output:
```
   Item  Color_Blue  Color_Green  Color_Red
0     1           0            0          1
1     2           1            0          0
2     3           0            1          0
3     4           0            0          1
```

### Using Scikit‑Learn
```python
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse=False)
colors = [['Red'], ['Blue'], ['Green'], ['Red']]
encoded = encoder.fit_transform(colors)
print(encoded)
```

Output:
```
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]]
```

---

## ⚠️ Trade‑offs
- **Pros:** Removes false ordinality, makes categorical data usable in ML models, widely supported.
- **Cons:** Can create **many columns** if categories are numerous (high cardinality), leading to memory and performance issues.
- **Mitigation:** Use techniques like **hashing encoding** or **embedding layers** for large categorical sets.

---

## 🎯 Summary
One‑hot encoding transforms categorical features into binary vectors, ensuring models treat categories independently. It’s essential for datasets like movie genres, user demographics, or product types where categories have no natural order.  

👉 Since you’re working with **movie genres**, one‑hot encoding is exactly why your dataset has columns like `Action`, `Comedy`, `Drama` with 0/1 values — later combined into a readable string. Would you like me to show you how to **reverse one‑hot encoding back into a single categorical column** (e.g., “Action,Drama”) for easier human interpretation?

In [5]:
# Convert text into numbers

# ActionComedyRomance -> [1, 1, 0, 0, 0, 1, 0, 0, .....]

cv = CountVectorizer ()

movie_vectors = cv.fit_transform ( movies [ 'genres' ] )


In [6]:
# Compute similarity

similarity = cosine_similarity ( movie_vectors )

print ( similarity )

[[1.         0.         0.         ... 0.         0.57735027 0.        ]
 [0.         1.         0.57735027 ... 0.         0.         0.        ]
 [0.         0.57735027 1.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 1.         0.         0.70710678]
 [0.57735027 0.         0.         ... 0.         1.         0.        ]
 [0.         0.         0.         ... 0.70710678 0.         1.        ]]


In [9]:
# Recommendation system

def recommend ( movie_name ):

    movie_index = movies [ movies [ 'title' ] == movie_name ].index [0]

    distances = list ( enumerate ( similarity [ movie_index ] ) )

    distances = sorted ( distances, key = lambda x : x [1], reverse = True ) 

    for i in distances [1:6]:

        if movies.iloc [ i [ 0 ] ][ 'title' ] != movie_name:

            print ( movies.iloc [ i[ 0 ] ][ 'title' ] )

print ( " --next : Batman Forever (1995) " )

recommend ( 'Batman Forever (1995)' )

print (" --next : Toy Story (1995)-- ")

recommend ( 'Toy Story (1995)' )

print (" --next : Titanic (1997)-- ")

recommend ( 'Titanic (1997)' )

print (" --next : Star Wars (1977)-- ")

recommend ( 'Star Wars (1977)' )

 --next : Batman Forever (1995) 
Batman Returns (1992)
Rumble in the Bronx (1995)
Batman & Robin (1997)
Three Musketeers, The (1993)
Cliffhanger (1993)
 --next : Toy Story (1995)-- 
Aladdin and the King of Thieves (1996)
Aladdin (1992)
Goofy Movie, A (1995)
Santa Clause, The (1994)
Home Alone (1990)
 --next : Titanic (1997)-- 
Man in the Iron Mask, The (1998)
Crying Game, The (1992)
First Knight (1995)
Postino, Il (1994)
 --next : Star Wars (1977)-- 
Return of the Jedi (1983)
Empire Strikes Back, The (1980)
Starship Troopers (1997)
African Queen, The (1951)
Stargate (1994)
